# AAA Clinical RAG — Final Evaluation

Evidence-first evaluation of a retrieval system over four abdominal aortic aneurysm (AAA)
clinical guidelines.

**How to read this notebook.** Every number is loaded from a committed artifact under `eval/`.
Nothing is recomputed here, so this notebook cannot disagree with the evaluation that produced
those artifacts. Three frozen question sets are reported **separately and never pooled**.

Three words are used precisely throughout:

| term | meaning |
|---|---|
| **we tried it** | the experiment ran and is preserved |
| **it worked** | it improved metrics on a frozen set it was measured against |
| **it generalized** | it improved metrics on a set that was frozen *before* the change existed |

Very little in this project reaches the third category. That is the honest result.

In [1]:
import json, hashlib, sys
from pathlib import Path
import numpy as np, pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 220)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EVAL = ROOT / "eval"

def load(p):
    return json.loads((ROOT / p).read_text(encoding="utf-8"))

RESULTS   = load("eval/final_evaluation_results.json")
EVIDENCE  = load("eval/final_evidence.json")
HISTORY   = load("eval/experiment_history.json")
CORPUS    = load("eval/corpus_audit.json")
QUESTIONS = load("eval/question_audit.json")
PROJECTB  = load("eval/project_b_comparison.json")
EXP12     = load("eval/runs/exp12_atomic_chunking.json")
INDEXMETA = load("data/embeddings/index_meta.json")
CHUNKS    = load("data/chunks/chunks.json")

try:
    STABILITY = load("eval/stability_report.json")
except FileNotFoundError:
    STABILITY = None

KEYS = ["P@1", "P@3", "P@5", "MRR", "Recall@5", "Recall@10", "Relevant_Top1", "Answering@5"]
DATASETS = ["original10", "heldout18", "final20"]

print("artifacts loaded")
for name, sha in RESULTS["gold_sha256"].items():
    print(f"  gold {name:<12} sha256 {sha}")

artifacts loaded
  gold original10   sha256 0b8a443b69960bc5ac20311f0010926a2f131bbb5531ccf369f321f59ed2e5c1
  gold heldout18    sha256 67112d0901337591c3e2d1c7f49bc532f800e5835084f6afeb9fa8b5ea8f1c82
  gold final20      sha256 36af11b88f7fd908fd474e904c41b567ef798b6608575f5302f3dee80f2e2579


## 1. Project overview

**Task.** Given a clinical question about abdominal aortic aneurysm, retrieve the guideline
passages that answer it, with full provenance (document, page, section, chunk id).

**Design commitments**, held throughout:

- Retrieval is **dense cosine similarity only**. No query rewriting, no intent detection, no
  keyword bonuses, no per-question rules. Nothing in the retrieval path can branch on *which*
  question is being asked — a rule that cannot see the question cannot be fitted to it.
- Every change is scored by a **frozen, retriever-agnostic evaluator** against pre-registered
  answer passages. Reverting is the normal outcome.
- Low-value material (references, contents pages, boilerplate) is **labelled and excluded from
  the index, never deleted** from `chunks.json`.

## 2. Corpus overview

In [2]:
docs = CORPUS["documents"]
print("pages per document :", docs["pages_per_document"])
print("total pages        :", sum(docs["pages_per_document"].values()))
print("chunks per document:", docs["chunks_per_document"])
print("indexed per doc    :", docs["indexed_per_document"])
print()
print(CORPUS["coverage_note"])

pages per document : {'ESVS_2024': 140, 'NICE_NG156': 53, 'SVS_2018': 48, 'USPSTF_2019': 8}
total pages        : 249
chunks per document: {'ESVS_2024': 1806, 'NICE_NG156': 158, 'SVS_2018': 48, 'USPSTF_2019': 104}
indexed per doc    : {'ESVS_2024': 1095, 'NICE_NG156': 118, 'SVS_2018': 44, 'USPSTF_2019': 73}

Four guidelines spanning the screening/diagnosis/surveillance/repair pathway from three regions (USPSTF 2019 US, NICE NG156 UK, ESVS 2024 Europe, SVS 2018 US). ESVS 2024 is the most recent and most detailed and supplies the majority of indexed chunks.


## 3. Guideline / document inventory

| id | guideline | year | region |
|---|---|---|---|
| `USPSTF_2019` | US Preventive Services Task Force — AAA screening recommendation statement | 2019 | US |
| `NICE_NG156` | NICE NG156 — Abdominal aortic aneurysm: diagnosis and management | 2020 | UK |
| `ESVS_2024` | ESVS 2024 Clinical Practice Guidelines — abdominal aorto-iliac artery aneurysms | 2024 | Europe |
| `SVS_2018` | Society for Vascular Surgery — AAA guideline (slide deck) | 2018 | US |

The corpus deliberately contains **genuinely conflicting recommendations**. A retrieval system
must surface all sides rather than silently pick one.

In [3]:
for c in CORPUS["known_conflicting_recommendations"]:
    print(f"* {c['topic']}  [{c['nature']}]")
    for s in c["sources"]:
        print("    -", s)

* Repair threshold in women  [Compatible in substance, different in wording and units.]
    - ESVS_2024 Recommendation 23 (p28): >= 50 mm may be considered
    - SVS_2018 (p19): repair suggested between 5.0 cm and 5.4 cm
* Preoperative beta blockers  [Genuinely different questions (starting vs continuing) that read as conflicting.]
    - NICE_NG156 1.4.7 (p15): do not routinely offer preoperative beta blockers
    - SVS_2018 (p13): continue beta blocker therapy if part of an established regimen
* Screening in women  [Genuine guideline disagreement. A retrieval system must surface all three, not pick one.]
    - USPSTF_2019: I statement / D recommendation depending on smoking history
    - NICE_NG156 1.1.3 (p7): consider aortic ultrasound for women 70+ with risk factors
    - SVS_2018 (p15): one-time screening in men OR women 65-75 with tobacco use


## 4. Chunk statistics

In [4]:
q = CHUNKS["quality"]
print(f"total chunks   : {q['total']}")
print(f"valid          : {q['valid']}    invalid: {q['invalid']}    duplicates: {q['duplicates']}")
print(f"status         : {q['status']}")
print(f"content types  : {q['content_types']}")
print(f"tokens         : {q['tokens']}")
print()
print("excluded from the index (labelled, not deleted):")
print(" ", CORPUS["low_value_material_excluded_from_index"])

total chunks   : 1760
valid          : 1760    invalid: 0    duplicates: 1
status         : PASS
content types  : {'clinical': 991, 'toc': 103, 'title_only': 26, 'reference': 634, 'boilerplate': 6}
tokens         : {'model_name': 'abhinand/MedEmbed-base-v0.1', 'token_limit': 512, 'n_chunks': 1760, 'min_tokens': 11, 'max_tokens': 512, 'mean_tokens': 212.07, 'median_tokens': 208.0, 'exceeding_limit': 0}

excluded from the index (labelled, not deleted):
  {'reference': 661, 'toc': 103, 'boilerplate': 14, 'title_only': 8, 'total_excluded': 786, 'policy': 'Labelled, retained in chunks.json for traceability, and filtered out of the index only.'}


## 5. Old vs final chunking

Baseline chunking is **page-driven**: a full guideline page already exceeds `TARGET_CHARS`, so
the buffer flushes at nearly every page end and 2,109 of 2,116 chunks are single-page. A
recommendation split by a page break is split across two chunks, and recommendation identity is
recovered only post hoc.

Experiment 12 chunking is **structure-driven**: the document is cut at structural anchors
(`Recommendation N`, numbered recommendation IDs, numbered section headings) and a recommendation
stays whole, under the same hard token budget.

**V1 is now what ships** (`clinical_chunking.DEFAULT_CHUNKER = "atomic"`,
`ingestion/atomic_chunking.py`). The table below is the Experiment 12 comparison; the
live index built from the shipped chunker is reported in sections 4 and 7.

In [5]:
prof = pd.DataFrame([{
    "variant": n,
    "chunks": v["chunk_profile"]["total_chunks"],
    "indexed": v["chunk_profile"]["indexed_chunks"],
    "mean tok": v["chunk_profile"]["tokens"]["mean"],
    "max tok": v["chunk_profile"]["tokens"]["max"],
    "over limit": v["chunk_profile"]["tokens"]["over_limit"],
    "pages/chunk": v["page_span_profile"]["mean_pages_per_chunk"],
    "% multipage": v["page_span_profile"]["pct_multi_page"],
    "with rec_id": v["chunk_profile"]["with_recommendation_id"],
} for n, v in EXP12["variants"].items()])
print(prof.to_string(index=False))

            variant  chunks  indexed  mean tok  max tok  over limit  pages/chunk  % multipage  with rec_id
 control_production    1330     1330     191.6      254           0        1.005          0.5          172
 V1_atomic_pagesafe    1764     1004     221.4      512           0        1.232         14.8          350
     V2_atomic_pure    1825     1081     217.1      512           0        5.568         81.6          376
    V3_size_control    2034     1259     199.8      503           0        1.003          0.3          146
V4_pagespan_control    1330     1330     191.6      254           0        4.960        100.0          172

## 6. Token-budget validation

Chunk size is budgeted in **tokens**, measured with the tokenizer of the model that will actually
encode the text — not in characters. `embeddable_chunks` **raises** rather than letting an
oversized chunk be silently truncated at encode time.

This is the single clearest engineering difference from Project B, which enforces no token budget
and silently discards 83,516 tokens across 143 of its 452 indexed chunks.

In [6]:
sys.path.insert(0, str(ROOT))
import ingestion.chunking as cc

idx = load("data/embeddings/embedded_chunks.json")
limit = INDEXMETA["token_limit"]
counts = [int(c["token_count"]) for c in idx]
print(f"model            : {INDEXMETA['model_name']}")
print(f"token limit      : {limit}")
print(f"max chunk tokens : {max(counts)}")
print(f"over the limit   : {sum(1 for t in counts if t > limit)}")
print()
b = PROJECTB["measured_project_b_facts"]
print("Project B, for contrast:")
print(f"  window {b['encoder_token_window']}, max chunk {b['indexed_token_max']}, "
      f"over window {b['indexed_chunks_over_token_window']}/{b['indexed_chunks']} "
      f"({b['percent_indexed_over_token_window']}%), "
      f"{b['tokens_silently_dropped_at_encode_time']} tokens dropped")

model            : abhinand/MedEmbed-base-v0.1
token limit      : 512
max chunk tokens : 512
over the limit   : 0

Project B, for contrast:
  window 256, max chunk 23079, over window 143/452 (31.6%), 83516 tokens dropped


## 7. Embedding / index statistics

In [7]:
vecs = np.load(ROOT / "data/embeddings/embeddings.npy")
norms = np.linalg.norm(vecs, axis=1)
print(f"model        : {INDEXMETA['model_name']}")
print(f"revision     : {INDEXMETA['model_revision']}   (pinned by commit)")
print(f"dimensions   : {INDEXMETA['embedding_dim']}")
print(f"vectors      : {vecs.shape}  {vecs.dtype}")
print(f"normalisation: L2, min {norms.min():.8f} max {norms.max():.8f}")
print(f"index type   : {INDEXMETA['index_type']} / {INDEXMETA['metric']}")

model        : abhinand/MedEmbed-base-v0.1
revision     : 7a90c50263f620dff743eb9794b89a42bfc5d765   (pinned by commit)
dimensions   : 768
vectors      : (991, 768)  float32
normalisation: L2, min 0.99999988 max 1.00000012
index type   : numpy_cosine / cosine


## 8. Retrieval methodology

```
query -> MedEmbed-base-v0.1 (pinned revision) -> L2-normalised vector
      -> cosine against 1,330 normalised chunk vectors (exhaustive)
      -> top-10, with document / page / section / chunk_id provenance
```

There is no second stage in the production path. The optional cross-encoder
(`retrieval/rerank.py`) exists but is **not wired in**, and the selective reranking
policy is experimental — see section 12 and `docs/limitations.md`.

## 9. Evaluation methodology

A retrieved chunk is **relevant** if and only if **both** hold:

1. **Provenance** — same `document_id`, and the chunk's `[page_start, page_end]` overlaps a
   pre-registered answer passage's page span.
2. **Facts** — the normalised chunk text satisfies at least `min_groups` of the query's
   `required_facts` groups.

No specific `chunk_id` is ever required, so the rule is independent of how the corpus is chunked.

**Three separate frozen sets.** They differ in difficulty and are never pooled.

In [8]:
rows = []
for ds in DATASETS:
    qa = QUESTIONS[ds]
    s = qa["summary"]
    rows.append({
        "dataset": ds, "questions": qa["n_questions"],
        "gold sha256": RESULTS["gold_sha256"][ds][:16] + "...",
        "answerable from corpus": f"{s['answerable_from_corpus']}/{qa['n_questions']}",
        "multi-fact": s["multi_fact"], "multi-document": s["multi_document"],
        "median relevant chunks in index": s["median_relevant_chunks_available"],
    })
print(pd.DataFrame(rows).to_string(index=False))
print()
print("Every question in all three sets is answerable from the index, so the measured")
print("ceiling is RETRIEVAL, not corpus coverage. final20 is the hardest by available evidence.")

   dataset  questions         gold sha256 answerable from corpus  multi-fact  multi-document  median relevant chunks in index
original10         10 0b8a443b69960bc5...                  10/10           2              10                               20
 heldout18         18 67112d0901337591...                  18/18           0              12                                7
   final20         20 36af11b88f7fd908...                  20/20           1               7                                4

Every question in all three sets is answerable from the index, so the measured
ceiling is RETRIEVAL, not corpus coverage. final20 is the hardest by available evidence.


## 10-12. Results on the three frozen sets

> **Historical labels.** These three tables were produced **before** V1 was promoted.
> `baseline_production` is the *old* page-buffer chunker, preserved in
> `data/archive_baseline_index/`; `V1_atomic_pagesafe` is what now ships (measured here with the
> citation-heading fix off — section 17a shows it makes no difference). They are kept exactly as
> produced, so the comparison that drove the decision stays reproducible.

**The retriever is identical in all three rows** — same model, same pinned revision, same dense
cosine, no reranking. The only variable is where chunk boundaries fall.

In [9]:
def table(ds):
    base = RESULTS["metrics"][ds]["baseline_production"]["metrics"]
    df = pd.DataFrame([{"config": c, **{k: m["metrics"][k] for k in KEYS}}
                       for c, m in RESULTS["metrics"][ds].items()])
    d = pd.DataFrame([{"config": "delta " + c,
                       **{k: round(m["metrics"][k] - base[k], 4) for k in KEYS}}
                      for c, m in RESULTS["metrics"][ds].items() if c != "baseline_production"])
    print(f"\n=== {ds} ===")
    print(df.to_string(index=False))
    print()
    print(d.to_string(index=False))

for ds in DATASETS:
    table(ds)


=== original10 ===
             config  P@1    P@3  P@5    MRR  Recall@5  Recall@10  Relevant_Top1  Answering@5
baseline_production  0.5 0.3333 0.40 0.6194    0.3567     0.4683              5            8
 V1_atomic_pagesafe  0.6 0.4333 0.40 0.7750    0.4150     0.5583              6           10
     V2_atomic_pure  0.7 0.5333 0.42 0.8200    0.3450     0.6183              7           10

                  config  P@1  P@3  P@5    MRR  Recall@5  Recall@10  Relevant_Top1  Answering@5
delta V1_atomic_pagesafe  0.1  0.1 0.00 0.1556    0.0583       0.09              1            2
    delta V2_atomic_pure  0.2  0.2 0.02 0.2006   -0.0117       0.15              2            2

=== heldout18 ===
             config    P@1    P@3    P@5    MRR  Recall@5  Recall@10  Relevant_Top1  Answering@5
baseline_production 0.5556 0.3704 0.3444 0.6972    0.6667     0.8241             10           16
 V1_atomic_pagesafe 0.7222 0.5556 0.4444 0.8148    0.7685     0.8981             13           17
     V2_a

                  config  P@1    P@3  P@5    MRR  Recall@5  Recall@10  Relevant_Top1  Answering@5
delta V1_atomic_pagesafe 0.15 0.0166 0.05 0.1330     0.125      0.175              3            2
    delta V2_atomic_pure 0.15 0.0666 0.08 0.1607     0.100      0.175              3            3


### Reading these three tables

`final20` is the only set that was **frozen before the configuration being tested existed**.
The original 10 and held-out 18 were both used to score V1/V2/V3/V4 during Experiment 12, so with
respect to the chunking decision they are selection sets, not held-out sets. Where the three
tables agree, the result generalized. Where they disagree, final20 is the one to believe.

## 13. Complete experiment history

In [10]:
hist = pd.DataFrame([{
    "#": e["n"], "phase": e["phase"], "experiment": e["name"][:52], "decision": e["decision"][:34],
} for e in HISTORY["experiments"]])
print(hist.to_string(index=False))

 #    phase                                           experiment                           decision
 1  Phase 0                   MiniLM baseline (all-MiniLM-L6-v2)                           BASELINE
 2  Phase 1         Experiment 1 - page-spanning recommendations                             REVERT
 3  Phase 1                Experiment 2 - header/footer cleaning                             REVERT
 4  Phase 1      Experiment 3 - improved section-title detection                             REVERT
 5  Phase 1 Experiment 4 - optional dense + cross-encoder rerank        KEEP AS OPT-IN, NOT DEFAULT
 6  Phase 2                      Experiment 5 - BGE-base-en-v1.5 NOT DEFAULT (validated optional co
 7  Phase 2     Experiment 6 - BGE-base + MS-MARCO cross-encoder                DO NOT MAKE DEFAULT
 8  Phase 2 Experiment 7 - MedEmbed-base-v0.1 (biomedical encode       ADOPTED - PRODUCTION DEFAULT
 9  Phase 3                               Q4 structural analysis        DIAGNOSTIC - no change made


In [11]:
# Full record for any single experiment.
def show(n):
    e = next(x for x in HISTORY["experiments"] if x["n"] == n)
    for f in ["name", "phase", "problem", "hypothesis", "intervention", "dataset",
              "evidence", "result", "failure_mode", "decision", "lesson_learned"]:
        print(f"{f.upper():<16} {e[f]}")
    print(f"{'METRICS':<16}")
    for k, v in e["metrics"].items():
        print(f"                 {k}: {v}")

show(21)   # the page-span control -- the experiment that changed the conclusion

NAME             Experiment 12 V4 - page-span control
PHASE            Phase 9
PROBLEM          The frozen relevance rule requires the chunk's page range to overlap the answer passage's page range, so wider page spans are easier to score relevant.
HYPOTHESIS       If page-span width alone can move the metrics, some of V2's advantage is measurement artifact rather than retrieval quality.
INTERVENTION     Took the PRODUCTION index unchanged - same vectors, same ranking, same retrieved chunks - and widened every chunk's page range by +/-2 pages to match V2's mean span. Nothing about retrieval changed.
DATASET          original10 and heldout18
EVIDENCE         eval/runs/exp12_atomic_chunking.json
RESULT           Widening page metadata ALONE, with identical retrieval, produced P@1 +0.10 (original10) and +0.1111 (heldout18), MRR +0.0667 / +0.0722.
FAILURE_MODE     None - this control worked exactly as intended, and it is the reason V2 was rejected.
DECISION         CONTROL - quantifies the 

## 14-15. Per-query evidence inspection

For every question, in every dataset, for every configuration: what was retrieved, at what rank,
with what similarity, from which document / page / section / chunk, whether the frozen rule
scored it relevant, which pre-registered answer passages it matched, and which required-fact
groups its text covers.

This table is sufficient to **reconstruct every metric** reported above.

In [12]:
EV = pd.DataFrame(EVIDENCE["rows"])
print(f"evidence rows: {len(EV)}  "
      f"({EV['dataset'].nunique()} datasets x {EV['config'].nunique()} configs x 10 ranks)")

def evidence_for(dataset, config, query_id, top=5):
    sel = EV[(EV.dataset == dataset) & (EV.config == config) & (EV.query_id == query_id)]
    print(f"\nQ{query_id} [{dataset} / {config}]")
    print(sel.iloc[0]["question"])
    cols = ["rank", "similarity", "document_id", "page_start", "page_end",
            "section_title", "relevant", "required_facts_covered", "chunk_id"]
    print(sel[cols].head(top).to_string(index=False))

evidence_for("final20", "baseline_production", 1)
evidence_for("final20", "V1_atomic_pagesafe", 1)

evidence rows: 1440  (3 datasets x 3 configs x 10 ranks)

Q1 [final20 / baseline_production]
At what maximum diameter is elective repair recommended for a man with an asymptomatic fusiform abdominal aortic aneurysm?
 rank  similarity document_id  page_start  page_end                                          section_title  relevant                            required_facts_covered                 chunk_id
    1    0.827368   ESVS_2024          27        27 There is anecdotal evidence that rapid aneurysm growth      True [repair, threshold_value, elective_or_indication] ESVS_2024__p27-27__c0355
    2    0.789685   ESVS_2024          27        27 There is anecdotal evidence that rapid aneurysm growth      True [repair, threshold_value, elective_or_indication] ESVS_2024__p27-27__c0354
    3    0.787651   ESVS_2024          10        10                                      Table 1-continued     False [repair, threshold_value, elective_or_indication] ESVS_2024__p10-10__c0181
    4    0.78326

 rank  similarity document_id  page_start  page_end                                          section_title  relevant                            required_facts_covered                 chunk_id
    1    0.826621   ESVS_2024          27        27 There is anecdotal evidence that rapid aneurysm growth      True [repair, threshold_value, elective_or_indication] ESVS_2024__p27-27__c0280
    2    0.821081   ESVS_2024          27        28 There is anecdotal evidence that rapid aneurysm growth      True [repair, threshold_value, elective_or_indication] ESVS_2024__p27-28__c0282
    3    0.807123   ESVS_2024          73        73    Complex AAAs are estimated to constitute about 15 e     False [repair, threshold_value, elective_or_indication] ESVS_2024__p73-73__c0665
    4    0.787651   ESVS_2024          10        10                                      Table 1-continued     False [repair, threshold_value, elective_or_indication] ESVS_2024__p10-10__c0133
    5    0.783106   ESVS_2024          2

In [13]:
# Expected evidence (pre-registered) vs what was actually retrieved, for one question.
gold20 = load("eval/gold_standard_final20.json")
spec = gold20["queries"][0]
print("QUESTION:", spec["query"])
print("\nEXPECTED EVIDENCE (pre-registered before any retrieval):")
for p in spec["answer_passages"]:
    print(f"  {p['document_id']} p{p['page_start']}-{p['page_end']}  [{p['section_ref']}]")
    print(f"     why: {p['why']}")
print("\nREQUIRED FACTS  (need >= "
      f"{spec['required_facts']['min_groups']} of {len(spec['required_facts']['groups'])} groups):")
for g in spec["required_facts"]["groups"]:
    print(f"  {g['name']}: {g['any_of']}")

QUESTION:

 At what maximum diameter is elective repair recommended for a man with an asymptomatic fusiform abdominal aortic aneurysm?

EXPECTED EVIDENCE (pre-registered before any retrieval):
  SVS_2018 p19-19  [Indications for repair]
     why: Elective repair recommended at >= 5.5 cm in low/acceptable-risk patients with a fusiform AAA.
  ESVS_2024 p27-27  [Recommendation 20]
     why: Men with an asymptomatic AAA < 55 mm are not recommended for elective repair.
  NICE_NG156 p16-16  [1.5.1 Repairing unruptured aneurysms]
     why: Consider repair if asymptomatic and larger than 5.5 cm.

REQUIRED FACTS  (need >= 2 of 3 groups):
  repair: ['repair']
  threshold_value: ['5\\.5\\s*cm', '55\\s*mm', '5\\.5']
  elective_or_indication: ['elective', 'indication', 'threshold', 'consider', 'recommend']


## 16. Failure analysis

In [14]:
rows = []
for ds in DATASETS:
    for cfg, m in RESULTS["metrics"][ds].items():
        for q in m["per_query"]:
            if q["first_relevant_rank"] is None:
                rows.append({"dataset": ds, "config": cfg, "query_id": q["query_id"],
                             "n_answer_passages": q["n_answer_passages"],
                             "top1_doc": q["top1_doc"]})
fail = pd.DataFrame(rows)
if len(fail):
    print("Questions with NO relevant chunk anywhere in the top 10:\n")
    print(fail.to_string(index=False))
    print()
    print("Counts by dataset/config:")
    print(fail.groupby(["dataset", "config"]).size().to_string())
else:
    print("No question failed completely in any dataset/configuration.")

Questions with NO relevant chunk anywhere in the top 10:

   dataset              config  query_id  n_answer_passages   top1_doc
original10 baseline_production         4                  5  ESVS_2024
 heldout18  V1_atomic_pagesafe        12                  2  ESVS_2024
 heldout18      V2_atomic_pure        12                  2  ESVS_2024
   final20 baseline_production         5                  1 NICE_NG156
   final20 baseline_production        11                  1 NICE_NG156
   final20 baseline_production        15                  1 NICE_NG156
   final20 baseline_production        16                  1  ESVS_2024
   final20 baseline_production        20                  1   SVS_2018
   final20  V1_atomic_pagesafe         5                  1  ESVS_2024
   final20  V1_atomic_pagesafe        16                  1  ESVS_2024
   final20      V2_atomic_pure         5                  1 NICE_NG156

Counts by dataset/config:
dataset     config             
final20     V1_atomic_pagesafe 

In [15]:
# Which questions did the chunking change RESCUE, and which did it BREAK?
for ds in DATASETS:
    base = {q["query_id"]: q for q in RESULTS["metrics"][ds]["baseline_production"]["per_query"]}
    for cfg in [c for c in RESULTS["metrics"][ds] if c != "baseline_production"]:
        cur = {q["query_id"]: q for q in RESULTS["metrics"][ds][cfg]["per_query"]}
        gained = [i for i in base if cur[i]["relevant_top1"] and not base[i]["relevant_top1"]]
        lost   = [i for i in base if base[i]["relevant_top1"] and not cur[i]["relevant_top1"]]
        print(f"{ds:<12} {cfg:<22} top-1 gained {gained}   top-1 LOST {lost}")

original10   V1_atomic_pagesafe     top-1 gained [10]   top-1 LOST []
original10   V2_atomic_pure         top-1 gained [6, 10]   top-1 LOST []
heldout18    V1_atomic_pagesafe     top-1 gained [2, 4, 14, 15]   top-1 LOST [9]
heldout18    V2_atomic_pure         top-1 gained [2, 4, 7, 14, 15]   top-1 LOST [9, 18]
final20      V1_atomic_pagesafe     top-1 gained [2, 11, 14, 15, 18, 20]   top-1 LOST [4, 7, 19]
final20      V2_atomic_pure         top-1 gained [2, 11, 14, 15, 18, 20]   top-1 LOST [3, 4, 19]


## 17. Chunking experiments — and the two controls that decide them

Two things can raise these metrics **without any retrieval improvement at all**: the relevance
rule rewards chunks that cover more text, and it requires page-range overlap. Both were
controlled.

- **V3 (size control)** — baseline algorithm, *no anchors*, budgets enlarged toward V1's mean.
- **V4 (page-span control)** — production index, *identical ranking and identical retrieved
  chunks*, page ranges widened to V2's mean span. Anything it gains is pure measurement artifact.
- **start-page-only scoring** — every chunk judged on its first page only. This removes the page
  term entirely and is a strict lower bound.

In [16]:
def exp12(key, title):
    base = EXP12["variants"]["control_production"][key]["metrics"]
    df = pd.DataFrame([{"variant": n, **{k: v[key]["metrics"][k] for k in KEYS}}
                       for n, v in EXP12["variants"].items()])
    print(f"\n=== {title} ===")
    print(df.to_string(index=False))

exp12("original_10", "original 10")
exp12("heldout_18", "held-out 18")
exp12("original_10_startpage_only", "original 10 - START PAGE ONLY (page confound removed)")
exp12("heldout_18_startpage_only", "held-out 18 - START PAGE ONLY (page confound removed)")


=== original 10 ===


            variant  P@1    P@3  P@5    MRR  Recall@5  Recall@10  Relevant_Top1  Answering@5
 control_production  0.5 0.3333 0.40 0.6194    0.3567     0.4683              5            8
 V1_atomic_pagesafe  0.6 0.4333 0.40 0.7750    0.4150     0.5583              6           10
     V2_atomic_pure  0.7 0.5333 0.42 0.8200    0.3450     0.6183              7           10
    V3_size_control  0.5 0.3000 0.38 0.6208    0.3317     0.4433              5            8
V4_pagespan_control  0.6 0.4000 0.44 0.6861    0.4417     0.5733              6            8

=== held-out 18 ===
            variant    P@1    P@3    P@5    MRR  Recall@5  Recall@10  Relevant_Top1  Answering@5
 control_production 0.5556 0.3704 0.3444 0.6972    0.6667     0.8241             10           16
 V1_atomic_pagesafe 0.7222 0.5556 0.4444 0.8148    0.7685     0.8981             13           17
     V2_atomic_pure 0.7222 0.6296 0.5222 0.8241    0.8519     0.8796             13           17
    V3_size_control 0.5556 0.425

**What the controls established.**

- **V4**: widening page metadata *alone*, with identical retrieval, gains **P@1 +0.10** (original
  10) and **+0.1111** (held-out 18). The confound is real and large.
- **V1** has a mean page span of 1.23 (baseline 1.005), so it is barely exposed. Its gains are
  **unchanged** under start-page-only scoring — the confound-free comparison.
- **V2** has a mean page span of 5.57 and 81.6% multi-page. Under start-page-only scoring its
  original-10 P@1 collapses from 0.70 to 0.40, *below* the 0.50 baseline. **The configuration
  with the best headline numbers has the weakest evidence.**
- **V3** is a *partial* size control: it reached a 199.8-token mean against V1's 221.4 (baseline
  191.6), covering roughly 28% of the size gap. The residual is not ruled out.

### 17a. FINAL CORRECTED VALIDATION — the promotion gate

Every V1/V2 number above was produced with a known anchor defect present: numbered *bibliography*
lines (`3 Svensjö S, Björck M, Gürtelschmid M, Djavani`) were being accepted as section headings.
The fix was gated off so the historical artifacts stayed reproducible.

The gate re-ran V1 with the fix **on**, against `final20` only, tuning nothing.

In [17]:
CORR = load("eval/runs/final_corrected_v1_final20.json")

cmp = pd.DataFrame([{
    "metric": k,
    "historical V1 (fix off)": CORR["metrics_historical_v1_fix_off"][k],
    "corrected V1 (fix ON, SHIPPED)": CORR["metrics_corrected"][k],
    "delta": CORR["delta"][k],
} for k in KEYS])
print(cmp.to_string(index=False))

print("\nanchors per document (without fix -> with fix):")
for doc in sorted(CORR["anchor_census"]["with_fix"]):
    w = CORR["anchor_census"]["with_fix"][doc]
    o = CORR["anchor_census"]["without_fix"][doc]
    print(f"  {doc:<14} sections {o.get('section',0):>4} -> {w.get('section',0):<4}"
          f"  recommendations {o.get('recommendation',0):>4} -> {w.get('recommendation',0)}")

hp, cp = CORR["chunk_profile_historical_v1"], CORR["chunk_profile_corrected"]
print(f"\n{'chunk stat':<26}{'historical':>12}{'corrected':>12}")
for lbl, key in [("total chunks","total_chunks"), ("indexed chunks","indexed_chunks"),
                 ("with recommendation_id","with_recommendation_id")]:
    print(f"{lbl:<26}{hp[key]:>12}{cp[key]:>12}")
for lbl, key in [("mean tokens","mean"), ("max tokens","max"), ("over model limit","over_limit")]:
    print(f"{lbl:<26}{hp['tokens'][key]:>12}{cp['tokens'][key]:>12}")

print(f"\nquestions whose rank/relevance changed: "
      f"{sum(1 for c in CORR['questions_with_any_change'] if c['historical_first_relevant_rank'] != c['corrected_first_relevant_rank'] or c['historical_relevant_top1'] != c['corrected_relevant_top1'])}")
print(f"questions whose top-1 CHUNK ID changed:  {CORR['n_questions_changed']}")
print(f"\nDECISION: {CORR['decision']}")

       metric  historical V1 (fix off)  corrected V1 (fix ON, SHIPPED)  delta
          P@1                   0.5500                          0.5500    0.0
          P@3                   0.3333                          0.3333    0.0
          P@5                   0.3000                          0.3000    0.0
          MRR                   0.6642                          0.6642    0.0
     Recall@5                   0.6667                          0.6667    0.0
    Recall@10                   0.7833                          0.7833    0.0
Relevant_Top1                  11.0000                         11.0000    0.0
  Answering@5                  16.0000                         16.0000    0.0

anchors per document (without fix -> with fix):
  ESVS_2024      sections  113 -> 100   recommendations  162 -> 162
  NICE_NG156     sections   14 -> 14    recommendations   57 -> 57
  SVS_2018       sections    0 -> 0     recommendations    0 -> 0
  USPSTF_2019    sections   15 -> 0     recommen

**All eight metrics identical (Δ 0.0000).** The fix removed 15 bogus USPSTF section anchors
(15 → 0) and 13 bogus ESVS ones (113 → 100); NICE and every recommendation anchor were untouched.
Seven of twenty questions returned a differently *identified* top-1 chunk — chunk IDs shift when
boundaries move — with unchanged rank and relevance. The removed anchors only ever fragmented
bibliography text, which the content classifier already excludes from the index.

A defect can be real, worth fixing, and still immaterial. Measuring exactly zero change is a
stronger statement than never having found it.

## 18. Retrieval-model comparison (all completed configurations)

In [18]:
hist_rows = []
for e in HISTORY["experiments"]:
    for ds, m in e["metrics"].items():
        if isinstance(m, dict):
            hist_rows.append({"experiment": e["name"][:46], "dataset/config": ds, **m})
        else:
            hist_rows.append({"experiment": e["name"][:46], "dataset/config": ds,
                              **{k: "n/a" for k in KEYS}})
comp = pd.DataFrame(hist_rows)
print(comp.to_string(index=False))
print()
print('"n/a" means: ' + HISTORY["unavailable_marker"])

                                    experiment                         dataset/config     P@1     P@3     P@5     MRR Recall@5 Recall@10 Relevant_Top1 Answering@5
            MiniLM baseline (all-MiniLM-L6-v2)                             original10     0.5  0.3667    0.28  0.5625   0.2217    0.2667             5           6
  Experiment 1 - page-spanning recommendations                             original10     0.5  0.3667    0.28  0.5611   0.2217    0.2667             5           6
         Experiment 2 - header/footer cleaning                             original10     0.5  0.3333    0.28  0.5625   0.2217    0.2917             5           6
Experiment 3 - improved section-title detectio                             original10     0.5  0.3667    0.28  0.5611   0.2217    0.2667             5           6
Experiment 4 - optional dense + cross-encoder              original10 (30 candidates)     0.5  0.3333    0.24  0.5736   0.1817    0.2917             5           6
Experiment 4 - optiona

## 19. Stability and reproducibility

In [19]:
if STABILITY is None:
    print("eval/stability_report.json not present -- run: python eval/run_stability_checks.py")
else:
    print("summary:", STABILITY["summary"])
    st = pd.DataFrame([{"check": c["check"][:66], "category": c["category"],
                        "status": c["status"], "sec": c["seconds"]}
                       for c in STABILITY["checks"]])
    print()
    print(st.to_string(index=False))

summary: {'pass': 14, 'fail': 1, 'not_run': 1}

                                                             check        category  status   sec
Chunking is deterministic: two rebuilds from the same inputs agree reproducibility    pass 58.20
      Committed chunks.json still reproduces from the CURRENT code reproducibility    pass  0.05
                        Chunk IDs unique and deterministic in form reproducibility    pass  0.05
                                Embedding model revision is pinned reproducibility    pass  0.00
Index integrity: vectors align with metadata and are L2-normalised reproducibility    pass  0.03
Index reproducibility: re-embedding a sample reproduces stored vec reproducibility    pass 40.87
         Token safety: no indexed chunk exceeds the encoder window     correctness    pass  2.01
              Fail-loud token validator rejects an oversized chunk     correctness    pass  0.00
          Malformed and adversarial queries do not crash retrieval      robustn

## 20. Runtime / performance

In [20]:
if STABILITY:
    for c in STABILITY["checks"]:
        if c["category"] == "performance":
            print(c["check"])
            for k, v in (c["detail"] or {}).items():
                print(f"   {k}: {v}")
            print()

Retrieval latency
   queries_timed: 10
   mean_seconds: 0.0719
   median_seconds: 0.0716
   p90_seconds: 0.0789
   max_seconds: 0.0893
   hardware: CPU only (torch CPU build); no GPU used
   note: Dominated by query encoding, not by the 1,330-vector dot product.

Index footprint on disk and in memory
   vectors_file_bytes: 3044480
   chunk_metadata_file_bytes: 2058057
   vectors_in_memory_bytes: 3044352
   shape: [991, 768]
   index_type: in-process NumPy matrix, exhaustive cosine (no ANN service)



## 21. Limitations

Full list: `docs/limitations.md`. The ones that most affect how these results should be read:

1. **10 / 18 / 20 questions.** One question is 0.10 of P@1 on the original 10. No significance
   is claimed anywhere.
2. **The relevance rule rewards page-span width** — measured at roughly +0.10 P@1 for free (V4).
3. **V3 controls chunk size only partially** (~28% of the gap).
4. **Anchors exist in only 2 of the 4 documents**; USPSTF and SVS fall back to baseline splitting.
5. **A known anchor defect (bibliography lines accepted as headings) is fixed in code but was
   present in every V1/V2 number here**, gated off so the artifacts still reproduce.
6. **Retrieval only** — no answer generation, no citation-faithfulness, no clinical safety
   evaluation. The corpus contains conflicting recommendations that the system surfaces
   without reconciling.
7. **Selective reranking Policy A remains INCONCLUSIVE** and was deliberately excluded from
   final20 so that final20 measures the chunking question alone.

## 22. Final recommendation

In [21]:
print(HISTORY.get("final_recommendation", "See docs/experiment_history.md"))

### Final decision: **ADOPT WITH CAVEATS** — V1 is the shipped chunker

**Shipped:** `V1_atomic_pagesafe` — structural-anchor chunk boundaries with recommendations kept
whole, page breaks still cutting narrative, under the existing hard token budget, with
`REJECT_CITATION_HEADINGS = True`.
**Rejected:** `V2_atomic_pure`.
**Unchanged:** the retriever. MedEmbed-base-v0.1 at pinned revision, dense cosine, top-10, no
reranking, no query processing.

#### Why V1

It is the only change in the project's history to raise P@1, and the only one to do so on a set
frozen **before the change existed**:

| set | P@1 base → V1 | MRR base → V1 | Recall@10 base → V1 | status of the set |
|---|---|---|---|---|
| original10 | 0.500 → **0.600** | 0.619 → **0.775** | 0.468 → **0.558** | used to select among variants |
| heldout18 | 0.556 → **0.722** | 0.697 → **0.815** | 0.824 → **0.898** | used to select among variants |
| **final20** | 0.400 → **0.550** | 0.531 → **0.664** | 0.608 → **0.783** | **pre-reg

## 23. Deployment readiness

See `docs/deployment_readiness.md` and `eval/stability_report.json`.

**Verified:** deterministic chunking, deterministic and unique chunk IDs, a pinned embedding
revision, index/metadata alignment with L2-normalised vectors, index reproducibility by
re-embedding, token safety with a fail-loud validator, robustness to malformed and adversarial
queries, ranking determinism, latency and footprint, and the test suite.

**Not production ready, and not claimed to be.** The repository ships no service, no API, no
logging, no authentication, no monitoring, and **no abstention threshold** — an empty or wholly
out-of-scope query still returns 10 chunks. Clean-checkout reproducibility *from the source PDFs*
was not verified, because that would destroy the artifacts every preserved evaluation is scored
against.

It is a reproducible research pipeline, which is what it was built to be.

In [22]:
if STABILITY:
    print("production_ready:", STABILITY["production_ready"])
    print()
    print(STABILITY["production_readiness_statement"])

production_ready: False

NOT production ready, and not claimed to be. The retrieval core is deterministic, reproducible and token-safe, but the repository ships no service, no API, no logging, no abstention threshold for out-of-scope queries, no authentication and no monitoring. It is a reproducible research pipeline, which is what it was built to be.
